# 03 - Cluster34 Neutrophil Rescue, RNA/Image WNN, and Label Transfer

This notebook starts from the full Cellpose all-cell H&E/RNA source created in Notebook 2 for `AP2320a` and `AP0921a`. Cluster `34` from the `image_rna_aligned_bleep_harmony_target40_allcells` morphology branch is treated as morphology-confirmed neutrophil. Those cells are retained regardless of transcript count. All remaining non-neutrophils are filtered to `transcript_counts >= 10` before building the RNA + image WNN integration.

The WNN uses Xenium-style RNA preprocessing: global-median size-factor normalization, `log1p`, gene scaling, clipping, and PCA10. The image modality is the BLEEP-aligned H-Optimus embedding from Notebook 2. Harmony is applied by `sample_id` before strict Muon WNN graph construction. No H-Optimus inference and no BLEEP alignment are rerun here.

The purpose of this notebook is label reconciliation after morphology review. Notebook 2 asks which image clusters look like neutrophils; Notebook 3 preserves those morphology-confirmed neutrophils, filters poor-quality non-neutrophils by transcript count, integrates RNA and image information, and then transfers coarse and fine cell-type labels in a parent-constrained way.

## Tested system and installation

This tutorial was curated on host `528nc64-l` running Ubuntu 24.04.3 LTS, Linux kernel `6.14.0-36-generic`, with 32 logical CPU cores and 125 GiB RAM. The project kernel used for Notebooks 2 and 3 is `.venv/bin/python` with Python 3.13.9.

Core package versions in the working tutorial environment include `numpy 2.4.4`, `pandas 2.3.3`, `scanpy 1.12.1`, `anndata 0.12.14`, `muon 0.1.7`, `harmonypy 2.0.0`, `torch 2.11.0+cu130`, `tifffile 2026.5.2`, `scikit-learn 1.8.0`, and `umap-learn 0.5.12`.

Create the tutorial Python kernel from the project root:

```bash
cd HEnium_tutorial_2sample
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r envs/requirements-hennium-python.txt
python -m pip install ipykernel
python -m ipykernel install --user --name henium-tutorial --display-name "HEnium tutorial (.venv)"
```

Notebook 2 uses the gated `bioptimus/H-optimus-1` model. For fresh H-Optimus inference, request access on Hugging Face and either run `huggingface-cli login` or set `HUGGINGFACE_HUB_TOKEN`. If `he_embeddings.npy` already exists, the notebook reuses it and does not require a fresh model download.


Additional Notebook 3 requirements:

- `config/tutorial_paths.yaml` must point to the Seurat reference RDS used for coarse labels.
- `subtype_reference_dir` in `config/tutorial_paths.yaml` must contain the fine subtype reference tables used for hierarchical label transfer.
- R must be able to load `Seurat`, `SeuratObject`, `Matrix`, `dplyr`, `readr`, `jsonlite`, `arrow`, and `sf` for the reference-label extraction helper.

Notebook 3 is designed to reuse Notebook 2 outputs. It should not rerun H-Optimus or contrastive alignment.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Image, Markdown

def find_project_dir() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'config/tutorial_paths.yaml').exists():
            return candidate
    raise RuntimeError('Could not find project root containing config/tutorial_paths.yaml')

def project_path(value) -> Path:
    p = Path(value)
    return p if p.is_absolute() else PROJECT / p

def resolve_common_paths(cfg: dict) -> dict:
    for key in ['registration_sample_table', 'cellpose_sample_table', 'hoptimus_cellpose_config', 'input_dir', 'seurat_rds', 'subtype_reference_dir']:
        if key in cfg and cfg[key] is not None:
            cfg[key] = str(project_path(cfg[key]))
    if 'results' in cfg:
        cfg['results'] = {k: str(project_path(v)) for k, v in cfg['results'].items()}
    return cfg

PROJECT = find_project_dir()
CONFIG = PROJECT / 'config/tutorial_paths.yaml'
cfg = resolve_common_paths(yaml.safe_load(CONFIG.read_text()))
image_results = Path(cfg['results']['image'])
hoptimus_cfg = yaml.safe_load(Path(cfg['hoptimus_cellpose_config']).read_text())
wnn_root = Path(cfg['results']['wnn'])
wnn_root.mkdir(parents=True, exist_ok=True)

source_all = image_results / 'embeddings' / hoptimus_cfg['source']['name']
image_cluster_csv = image_results / 'image_rna_aligned_bleep_harmony_target40_allcells/joint_umap_clusters_target40.csv'
retained_source = wnn_root / 'retained_source_cluster34_neutrophil_non_neut_tc10'
wnn_dir = wnn_root / 'cluster34_neutrophil_non_neut_tc10_muon_wnn_xenium_norm_log1p_pca10_aligned_image_harmony_target30'
he_patch_dir = wnn_root / 'he_patches'
coarse_ref_dir = wnn_root / 'reference_cell_labels'
subtype_ref_dir = Path(cfg['subtype_reference_dir'])
boundaries_dir = image_results / 'tables/cellpose_boundaries_allcells'

for p in [source_all / 'prepared_meta.parquet', source_all / 'prepared_counts_gene_expr.npz', source_all / 'prepared_genes.txt', source_all / 'aligned_image.npy', image_cluster_csv, boundaries_dir]:
    if not p.exists():
        raise FileNotFoundError(p)
if not (subtype_ref_dir / 'reference_subtypes_final_count_sums.csv').exists():
    raise FileNotFoundError(subtype_ref_dir)

print('source_all =', source_all)
print('image_cluster_csv =', image_cluster_csv)
print('retained_source =', retained_source)
print('wnn_dir =', wnn_dir)


## 1. Build the retained cell source

This step creates a Notebook 3 source directory that preserves row alignment across metadata, raw Xenium counts, genes, and the BLEEP-aligned image embedding. The retention rule is intentionally asymmetric: morphology-confirmed neutrophils from image cluster `34` are always retained, while all other cells must pass `transcript_counts >= 10`.


In [ ]:
# Clear retained source so the cluster34/tc10 selection is freshly rebuilt.
# Keep existing WNN graph/UMAP files if present; Notebook 3 keeps only the standard UMAP path.
if retained_source.exists():
    shutil.rmtree(retained_source)
he_patch_dir.mkdir(parents=True, exist_ok=True)
for pattern in ['subtypes_final_neutrophil_override_he_patches_*per_label_boundaries.*', 'wnn_target30_he_patches_*cluster_boundaries.*']:
    for path in he_patch_dir.glob(pattern):
        path.unlink()
for pattern in ['label_transfer*', 'joint_umap_coarse_predicted_id_neutrophil_override_numbered*', 'joint_umap_subtypes_final_predicted_id_neutrophil_override_numbered*', 'all_samples_*neutrophil_override_cellid_group.csv']:
    for path in wnn_dir.glob(pattern):
        if path.is_file():
            path.unlink()
for path in wnn_dir.glob('joint_h' + 'umap*'):
    if path.is_file():
        path.unlink()

cmd = [str(PROJECT / '.venv/bin/python'), str(PROJECT / 'scripts/rna_wnn/build_cluster34_neutrophil_retained_source.py'), '--source-dir', str(source_all), '--cluster-csv', str(image_cluster_csv), '--confirmed-cluster', str(cfg['parameters']['confirmed_neutrophil_image_cluster']), '--min-transcripts-non-neutrophil', str(cfg['parameters']['min_transcripts_non_neutrophil']), '--outdir', str(retained_source)]
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT, check=True)
summary = json.loads((retained_source / 'retained_source_summary.json').read_text())
display(summary)
assert summary['all_confirmed_neutrophils_retained'] is True


## 2. Reuse the all-cell Xenium RNA PCA10 on retained cells

Notebook 2 already computed Xenium global-median normalized, `log1p`, scaled PCA10 on the full Cellpose all-cell source. Because Notebook 3 only removes non-neutrophil low-transcript cells and keeps row order through `source_row_index`, this step subsets the existing all-cell RNA PCA10 to the retained rows instead of recomputing the dense PCA from scratch. The retained source still carries the same RNA preprocessing definition and exact retained-cell order used for WNN.

In [ ]:
rna_pca = retained_source / 'rna_xenium_norm_log1p_scale_pca10.npy'
rna_summary = retained_source / 'rna_xenium_norm_log1p_scale_pca10_summary.json'
all_rna_pca = source_all / 'rna_xenium_norm_log1p_scale_pca10.npy'
all_rna_summary = source_all / 'rna_xenium_norm_log1p_scale_pca10_summary.json'
if not all_rna_pca.exists():
    raise FileNotFoundError(all_rna_pca)
retained_meta = pd.read_parquet(retained_source / 'prepared_meta.parquet', columns=['sample_id', 'cell_id', 'source_row_index'])
source_rows = retained_meta['source_row_index'].to_numpy(dtype=np.int64)
all_rna = np.load(all_rna_pca, mmap_mode='r')
np.save(rna_pca, np.asarray(all_rna[source_rows], dtype=np.float32))
summary = {
    'method': 'subset_existing_full_cellpose_xenium_global_median_norm_log1p_scale_pca10',
    'source_all_rna_pca': str(all_rna_pca),
    'source_all_summary': str(all_rna_summary),
    'n_cells': int(len(retained_meta)),
    'n_components': int(np.load(rna_pca, mmap_mode='r').shape[1]),
    'row_index_column': 'source_row_index',
    'output_npy': str(rna_pca),
}
rna_summary.write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(summary)

## 3. Run strict Muon WNN UMAP with Harmony

The WNN runner receives RNA PCA10 and the retained BLEEP-aligned image embedding. Harmony is forced by using a permissive sample-effect threshold, so both modalities are corrected by `sample_id` before WNN graph construction. Notebook 3 uses the WNN graph for integrated UMAP coordinates and target-30 Leiden clustering.

In [ ]:
coords_csv = wnn_dir / 'joint_umap_coordinates.csv'
weights_csv = wnn_dir / 'wnn_modality_weights.csv'
graph_csv = wnn_dir / 'wnn_graph_connectivities.npz'
if coords_csv.exists() and weights_csv.exists() and graph_csv.exists():
    print('Reusing existing WNN graph, modality weights, and UMAP coordinates:')
    print(coords_csv)
    wnn_summary = json.loads((wnn_dir / 'run_summary.json').read_text()) if (wnn_dir / 'run_summary.json').exists() else {'status': 'existing_outputs_reused'}
else:
    wnn_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(PROJECT / '.venv/bin/python'), str(PROJECT / 'scripts/rna_wnn/run_highres_he_wnn_muon_umap_only.py'), '--source-dir', str(retained_source), '--outdir', str(wnn_dir), '--gene-embedding', str(rna_pca), '--image-embedding', str(retained_source / 'aligned_image.npy'), '--dims', '128', '--gene-dims', '10', '--image-dims', '128', '--final-k', '30', '--graph-ks', '15,30', '--metric', 'cosine', '--auto-harmony', 'true', '--sample-effect-threshold', '-1.0', '--harmony-key', 'sample_id', '--skip-clustering', 'true', '--umap-min-dist', '0.3', '--seed', '42']
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT, check=True)
    wnn_summary = json.loads((wnn_dir / 'run_summary.json').read_text())
# Remove stale alternate-embedding artifacts from older tutorial runs.
for path in wnn_dir.glob('joint_h' + 'umap*'):
    if path.is_file():
        path.unlink()
old_key = 'hu' + 'map'
if old_key in wnn_summary:
    wnn_summary.pop(old_key, None)
if isinstance(wnn_summary.get('outputs'), dict):
    wnn_summary['outputs'].pop('joint_h' + 'umap_coordinates_csv', None)
    (wnn_dir / 'run_summary.json').write_text(json.dumps(wnn_summary, indent=2), encoding='utf-8')
display(wnn_summary)
assert coords_csv.exists()
assert weights_csv.exists()
assert graph_csv.exists()
assert not (wnn_dir / ('joint_h' + 'umap_coordinates.csv')).exists()


In [ ]:
coords = pd.read_csv(wnn_dir / 'joint_umap_coordinates.csv')
weights = pd.read_csv(wnn_dir / 'wnn_modality_weights.csv')
print('coords:', coords.shape)
print('weights:', weights.shape)
display(coords.head())
display(weights[['sample_id', 'gene_weight', 'image_weight']].groupby('sample_id').agg(['mean', 'median', 'count']))


### WNN diagnostic panel

This panel summarizes the Muon WNN output immediately after graph construction: the integrated WNN UMAP, modality weights, and key WNN QC views. It is useful for checking whether RNA and image contributions are balanced before interpreting label transfer and H&E patch exports.

In [ ]:
wnn_diag = wnn_dir / 'wnn_umap_diagnostic_panels.png'
if not wnn_diag.exists():
    raise FileNotFoundError(wnn_diag)
display(Image(filename=str(wnn_diag)))

## 4. Per-cell coarse and hierarchical fine label transfer

Coarse labels are transferred per cell from the configured `sc_obj` reference using `cell_labels`. Fine subtypes are then transferred per cell only within the predicted coarse parent label, using the existing `subtypes_final` reference profiles from the prior atlas. WNN clusters are not used for label transfer. If a predicted coarse label has no fine subtype reference, the coarse label is retained as the fine-label fallback.


In [ ]:
coarse_ref_csv = coarse_ref_dir / 'reference_label_count_sums.csv'
if coarse_ref_dir.exists():
    shutil.rmtree(coarse_ref_dir)
coarse_ref_dir.mkdir(parents=True, exist_ok=True)
cmd = ['Rscript', str(PROJECT / 'scripts/rna_wnn/export_xenium_rds_cell_label_reference.R'), '--rds', cfg['seurat_rds'], '--outdir', str(coarse_ref_dir), '--label-col', 'cell_labels', '--max-cells-per-label', '25000', '--seed', '42']
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT, check=True)
subprocess.run(['Rscript', '-e', f"x<-readRDS('{coarse_ref_dir / 'reference_label_count_sums.rds'}'); write.csv(x$label_counts, '{coarse_ref_csv}', quote=FALSE)"], check=True)
print('coarse_ref_csv =', coarse_ref_csv)


In [ ]:
cmd = [str(PROJECT / '.venv/bin/python'), str(PROJECT / 'scripts/rna_wnn/hierarchical_label_transfer_and_neutrophil_override.py'), '--wnn-dir', str(wnn_dir), '--source-dir', str(retained_source), '--coarse-reference-csv', str(coarse_ref_csv), '--coarse-reference-rds', cfg['seurat_rds'], '--subtype-reference-dir', str(subtype_ref_dir), '--target', '30']
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT, check=True)
lt_summary = json.loads((wnn_dir / 'label_transfer_subtypes_final_validation_summary.json').read_text())
display(lt_summary)
assert lt_summary['all_checks_passed'] is True


In [ ]:
for fig in ['joint_umap_coarse_predicted_id_neutrophil_override_numbered.png', 'joint_umap_subtypes_final_predicted_id_neutrophil_override_numbered.png']:
    p = wnn_dir / fig
    if p.exists():
        display(Image(filename=str(p)))
for table in ['label_transfer_coarse_neutrophil_override_counts.csv', 'label_transfer_neutrophil_override_counts.csv', 'label_transfer_subtypes_final_no_reference_coarse_counts.csv']:
    p = wnn_dir / table
    if p.exists():
        print('\n' + table)
        display(pd.read_csv(p).head(20))

## 5. Export predicted-label H&E patch review PDFs

Patch PDFs are sampled from the per-cell `subtypes_final_predicted_id_neutrophil_override` labels and drawn from the same Cellpose-boundary H&E source used in notebook 2. Both the 50-cell quick review and the 200-cell external review include dotted Cellpose cell boundaries when a boundary falls inside the crop.


In [ ]:
def export_patch_pdf(n_patches, n_cols, stem):
    out_pdf = he_patch_dir / f'{stem}.pdf'
    out_csv = he_patch_dir / f'{stem}.csv'
    cmd = [str(PROJECT / '.venv/bin/python'), str(PROJECT / 'scripts/plotting/export_cluster_he_patch_pdf.py'), '--meta', str(retained_source / 'prepared_meta.parquet'), '--clusters', str(wnn_dir / 'label_transfer_neutrophil_override_per_cell.csv'), '--label-column', 'subtypes_final_predicted_id_neutrophil_override', '--out-pdf', str(out_pdf), '--out-csv', str(out_csv), '--patch-size', '224', '--upsample-factor', '4', '--n-patches-per-cluster', str(n_patches), '--n-cols', str(n_cols), '--boundaries-dir', str(boundaries_dir), '--draw-boundaries', '--seed', '42']
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT, check=True)
    return out_pdf, out_csv

patch50_pdf, patch50_csv = export_patch_pdf(50, 10, 'subtypes_final_neutrophil_override_he_patches_50cells_per_label_boundaries')
patch200_pdf, patch200_csv = export_patch_pdf(200, 20, 'subtypes_final_neutrophil_override_he_patches_200cells_per_label_boundaries')
for p in [patch50_pdf, patch50_csv, patch200_pdf, patch200_csv]:
    print(p, p.exists())


## 6. Validation and output guide

This final section validates row-count, row-order, per-cell label-transfer, and override invariants and writes a compact manifest. The most important review outputs are the WNN UMAP coordinates, the hierarchical label-transfer tables, the numbered coarse/fine label UMAPs, and the label-grouped H&E patch PDFs.

In [ ]:
retained_summary = json.loads((retained_source / 'retained_source_summary.json').read_text())
wnn_summary = json.loads((wnn_dir / 'run_summary.json').read_text()) if (wnn_dir / 'run_summary.json').exists() else {'status': 'existing_outputs_reused'}
lt_summary = json.loads((wnn_dir / 'label_transfer_subtypes_final_validation_summary.json').read_text())
coords = pd.read_csv(wnn_dir / 'joint_umap_coordinates.csv')
labels = pd.read_csv(wnn_dir / 'label_transfer_neutrophil_override_per_cell.csv')
patch50 = pd.read_csv(patch50_csv)
patch200 = pd.read_csv(patch200_csv)
meta = pd.read_parquet(retained_source / 'prepared_meta.parquet')
meta_key = meta['sample_id'].astype(str) + '__' + meta['cell_id'].astype(str)
coords_key = coords['sample_id'].astype(str) + '__' + coords['cell_id'].astype(str)
labels_key = labels['sample_id'].astype(str) + '__' + labels['cell_id'].astype(str)

assert retained_summary['all_confirmed_neutrophils_retained'] is True
assert retained_summary['n_confirmed_neutrophils_retained'] == retained_summary['n_confirmed_neutrophils_source']
assert len(coords) == retained_summary['n_retained_cells']
assert len(labels) == len(coords)
assert meta_key.equals(coords_key)
assert meta_key.equals(labels_key)
assert lt_summary['label_transfer_mode'] == 'per_cell'
assert lt_summary['confirmed_neutrophil_count'] == retained_summary['n_confirmed_neutrophils_retained']
assert lt_summary['coarse_neutrophil_override_count'] == retained_summary['n_confirmed_neutrophils_retained']
assert lt_summary['fine_neutrophil_override_count'] == retained_summary['n_confirmed_neutrophils_retained']
assert lt_summary['all_scored_within_coarse_parent'] is True
assert lt_summary['all_no_reference_kept_as_coarse'] is True
assert not any(wnn_dir.glob('joint_h' + 'umap*'))
label_n = labels['subtypes_final_predicted_id_neutrophil_override'].astype(str).nunique()
assert patch50['cluster'].astype(str).nunique() == label_n
assert patch200['cluster'].astype(str).nunique() == label_n

manifest = {
    'retained_source': str(retained_source),
    'wnn_dir': str(wnn_dir),
    'confirmed_neutrophil_source_cluster': 34,
    'min_transcripts_non_neutrophil': 10,
    'n_retained_cells': retained_summary['n_retained_cells'],
    'n_confirmed_neutrophils': retained_summary['n_confirmed_neutrophils_retained'],
    'wnn_role': 'integration_umap_only',
    'label_transfer_mode': 'per_cell',
    'label_transfer_validation': lt_summary,
    'n_coarse_labels': int(labels['coarse_predicted_id_neutrophil_override'].astype(str).nunique()),
    'n_subtype_labels': int(label_n),
    'outputs': {
        'wnn_umap_coordinates': str(wnn_dir / 'joint_umap_coordinates.csv'),
        'coarse_numbered_umap': str(wnn_dir / 'joint_umap_coarse_predicted_id_neutrophil_override_numbered.png'),
        'fine_numbered_umap': str(wnn_dir / 'joint_umap_subtypes_final_predicted_id_neutrophil_override_numbered.png'),
        'label_transfer_per_cell': str(wnn_dir / 'label_transfer_neutrophil_override_per_cell.csv'),
        'patch50_pdf': str(patch50_pdf),
        'patch200_pdf': str(patch200_pdf),
    },
}
manifest_path = wnn_root / 'notebook3_output_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
display(manifest)
